In [2]:
import pandas as pd

movies_df = pd.read_csv('./ml-latest-small/movies.csv')
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
ratings_df = pd.read_csv('./ml-latest-small/ratings.csv')
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [7]:
movies_df.isna().sum(), ratings_df.isna().sum()

(movieId    0
 title      0
 genres     0
 dtype: int64,
 userId       0
 movieId      0
 rating       0
 timestamp    0
 dtype: int64)

In [8]:
movies_df.duplicated().sum(), ratings_df.duplicated().sum()

(0, 0)

In [4]:
new_ratings_df = ratings_df.drop('timestamp', axis=1)
new_ratings_df.head()

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


In [6]:
new_ratings_df['rating'].value_counts().sort_index()

rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64

In [15]:
from surprise import Reader, Dataset

data = Dataset.load_from_df(new_ratings_df, Reader(rating_scale=(0, 5)))

In [20]:
train_data = data.build_full_trainset()

print('Number of users: ', train_data.n_users, '\n')
print('Number of items: ', train_data.n_items)

Number of users:  610 

Number of items:  9724


In [21]:
# Importing relevant libraries
from surprise.model_selection import cross_validate
from surprise.prediction_algorithms import SVD
from surprise.prediction_algorithms import KNNWithMeans, KNNBasic, KNNBaseline
from surprise.model_selection import GridSearchCV
import numpy as np

In [ ]:
## Perform a gridsearch with SVD
# This cell may take several minutes to run
param_grid = {
    'n_factors': [50, 100],
    'n_epochs': [20, 50],
    'lr_all': [0.005, 0.0005],
    'reg_all': [0.02, 0.04]
}
svd_model = GridSearchCV(
    algo_class=SVD,
    param_grid=param_grid,
    cv=3,
    refit=True,
    measures=['rmse']
)
svd_model.fit(data)

In [23]:
# Print out optimal parameters for SVD after GridSearch
print(svd_model.best_score['rmse'])
svd_model.best_params['rmse']

0.8711162344411439


{'n_factors': 100, 'n_epochs': 50, 'lr_all': 0.005, 'reg_all': 0.04}

In [24]:
# Cross validating with KNNBasic
param_grid = {
    'k': [40, 50],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True]
    }
}
knn_basic_model = GridSearchCV(
    algo_class=KNNBasic,
    param_grid=param_grid,
    refit=True,
    measures=['rmse'],
    cv=3
)
knn_basic_model.fit(data)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Comput

In [25]:
# Print out the average RMSE score for the test set
print(knn_basic_model.best_score)
knn_basic_model.best_params

{'rmse': 0.9752611472085325}


{'rmse': {'k': 40,
  'min_k': 5,
  'sim_options': {'name': 'cosine', 'user_based': True}}}

In [ ]:
# Cross validating with KNNBaseline
param_grid = {
    'k': [30, 40],
    'min_k': [5, 10],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True]
    }
}
knn_baseline_model = GridSearchCV(
    algo_class=KNNBaseline,
    param_grid=param_grid,
    refit=True,
    measures=['rmse'],
    cv=3
)
knn_baseline_model.fit(data)

Estimating biases using als...
Estimating biases using als...
Computing the cosine similarity matrix...
Computing the cosine similarity matrix...
Estimating biases using als...
Computing the cosine similarity matrix...
Estimating biases using als...
Done computing similarity matrix.
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.
Done computing similarity matrix.
Estimating biases using als...
Estimating biases using als...
Computing the cosine similarity matrix...
Computing the pearson similarity matr

In [27]:
# print out the average score for the test set
print(knn_baseline_model.best_score)
knn_baseline_model.best_params

{'rmse': 0.872756864901826}


{'rmse': {'k': 40,
  'min_k': 10,
  'sim_options': {'name': 'pearson', 'user_based': True}}}

In [28]:
# Cross validating with KNNWithMeans
param_grid = {
    'k': [30, 40],
    'min_k': [5, 10],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True]
    }
}
knn_with_means_model = GridSearchCV(
    algo_class=KNNWithMeans,
    param_grid=param_grid,
    refit=True,
    measures=['rmse'],
    cv=3
)
knn_with_means_model.fit(data)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Comput

In [ ]:
# print out the average score for the test set
print(knn_with_means_model.best_score)
knn_with_means_model.best_params